# Лекция 10 · Перенос наночастиц: конвекция, диффузия и внешние поля

**Исполняемый учебный пример для Au, Fe₃O₄ и SiO₂.** Все величины в формулах и коде заданы в СИ; в таблицах и рисунках используются мкм и секунды. Нужны только `numpy` и `matplotlib` из окружения курса. Запустите все ячейки последовательно из чистого ядра.

Цель: показать, что диффузию задают *гидродинамический* размер, температура и вязкость; плотность определяет оседание; измеренный ζ-потенциал может влиять на электрофорез только вместе с подходящей моделью подвижности; магнитный градиент действует на магнитный материал. Все численные входы здесь **иллюстративные**, а не результаты измерения этих частиц. Проверка моментов симуляции против аналитического ответа проверяет код, но не подтверждает применимость модели к конкретной дисперсии.

## Паспорт и границы

Каждая `System` хранит материал, диаметр в нм, ζ в мВ, объёмную долю этанола, температуру, динамическую вязкость, плотность и диэлектрическую проницаемость **заданной среды**, а также ионную силу. ζ зависит от состава, pH и способа измерения и не является свойством неизменного «ядра». Значения среды при изменении доли этанола или температуры **надо задавать заново**: здесь нет линейной интерполяции вязкости или ζ. `Fields` задаёт скорость жидкости, локальный сдвиг, электрическое поле, магнитную индукцию и её градиент.

Предположения: разбавленная устойчивая дисперсия одинаковых невзаимодействующих гладких сфер вдали от стенок; ньютоновская среда с постоянными свойствами; изотермический режим; нет градиента концентрации соли, испарения, агрегации и парных гидродинамических взаимодействий. Координата $y$ направлена вверх. В одночастичной задаче ζ **не меняет** $D$ при выключенном электрическом поле: электростатическое межчастичное отталкивание в этой модели отсутствует. Плотность относится к эффективной *сплошной* сфере с указанным гидродинамическим диаметром; для оболочек и пористых агрегатов понадобится отдельная плавучая масса.

Уравнение Эйлера–Маруямы для каждой частицы:

$$\Delta x=[u_0+\dot\gamma y+\mu_{ep} E+F_m/(6\pi\eta a)]\Delta t+\sqrt{2D\Delta t}\,\xi_x,$$
$$\Delta y=v_g\Delta t+\sqrt{2D\Delta t}\,\xi_y,$$

где $a=d_h/2$, $D=k_BT/(6\pi\eta a)$, $v_g=-2(\rho_p-\rho_m)ga^2/(9\eta)$, независимые $\xi_x,\xi_y\sim N(0,1)$. При $\dot\gamma=0$ и постоянных коэффициентах $\langle\Delta\mathbf r\rangle=\mathbf v t$, а центрированный двумерный $\mathrm{MSD}=4Dt$. Не вычитайте лишь заданный поток при проверке MSD в полях: электрический и магнитный дрейф тоже смещают среднее.

In [ ]:
from dataclasses import dataclass, replace

import matplotlib.pyplot as plt
import numpy as np

k_B = 1.380649e-23             # Дж/К
epsilon_0 = 8.8541878128e-12  # Ф/м
mu_0 = 1.25663706212e-6       # Н/А²
e_charge = 1.602176634e-19    # Кл
N_A = 6.02214076e23           # моль⁻¹
g = 9.80665                    # м/с²


@dataclass(frozen=True)
class System:
    material: str
    diameter_nm: float              # гидродинамический диаметр, нм
    zeta_mV: float                  # измеряется для ЭТОЙ среды и T
    ethanol_frac: float             # объёмная доля этанола при приготовлении
    T_K: float
    eta_Pa_s: float                 # введённая отдельно вязкость состава при T
    rho_medium_kg_m3: float         # введённая отдельно плотность состава при T
    epsilon_r: float                # введённая отдельно относительная проницаемость
    ionic_strength_M: float         # моль/л, расчёт λ_D только для водного 1:1 электролита
    rho_particle_kg_m3: float       # эффективная плотность сплошной сферы
    chi_eff: float = 0.0            # эффективная объёмная восприимчивость для Fe₃O₄
    measured_mu_ep_m2_V_s: float | None = None  # при наличии измеренной подвижности


@dataclass(frozen=True)
class Fields:
    u0_m_s: float = 0.0
    shear_s_inv: float = 0.0
    E_x_V_m: float = 0.0
    B_T: float = 0.0
    gradient_T_m: float = 0.0


def check_inputs(s: System):
    assert 0 <= s.ethanol_frac <= 1
    assert s.diameter_nm > 0 and s.T_K > 0 and s.eta_Pa_s > 0
    assert s.rho_medium_kg_m3 > 0 and s.rho_particle_kg_m3 > 0
    assert s.epsilon_r > 0 and s.ionic_strength_M >= 0 and s.chi_eff >= 0


def hydrodynamic_radius_m(s: System):
    return s.diameter_nm * 0.5e-9


def diffusion_m2_s(s: System):
    check_inputs(s)
    return k_B * s.T_K / (6 * np.pi * s.eta_Pa_s * hydrodynamic_radius_m(s))


def debye_length_m(s: System):
    # Идеальный 1:1 водный электролит; I в моль/л → моль/м³.
    if s.ethanol_frac != 0 or s.ionic_strength_M <= 0:
        raise ValueError("λ_D здесь определена только для водного 1:1 электролита с I>0")
    return np.sqrt(
        epsilon_0 * s.epsilon_r * k_B * s.T_K
        / (2 * N_A * e_charge**2 * 1000 * s.ionic_strength_M)
    )


def electrophoretic_mobility_m2_V_s(s: System):
    if s.measured_mu_ep_m2_V_s is not None:
        return s.measured_mu_ep_m2_V_s, "измеренная μ"
    if s.ethanol_frac != 0:
        raise ValueError("Для водно-спиртовой среды укажите measured_mu_ep_m2_V_s")
    ratio = hydrodynamic_radius_m(s) / debye_length_m(s)
    if ratio < 20:
        raise ValueError(f"a/λ_D={ratio:.1f} < 20: приближение тонкого слоя не принято")
    # При κa≫1 и применимой модели тонкого двойного слоя; не Q/(6πηa).
    mu_ep = epsilon_0 * s.epsilon_r * s.zeta_mV * 1e-3 / s.eta_Pa_s
    return mu_ep, f"Смолуховский, a/λ_D={ratio:.1f}"


def coefficients(s: System, f: Fields):
    check_inputs(s)
    a = hydrodynamic_radius_m(s)
    V = 4 * np.pi * a**3 / 3
    D = diffusion_m2_s(s)
    v_g = -2 * (s.rho_particle_kg_m3 - s.rho_medium_kg_m3) * g * a**2 / (9 * s.eta_Pa_s)
    mu_ep, ep_method = (0.0, "E=0") if f.E_x_V_m == 0 else electrophoretic_mobility_m2_V_s(s)
    v_ep = mu_ep * f.E_x_V_m
    # Локальное линейное приближение M=χeff H, H≈B/μ0, без насыщения и взаимодействий.
    F_mag = V * s.chi_eff * f.B_T * f.gradient_T_m / mu_0
    v_mag = F_mag / (6 * np.pi * s.eta_Pa_s * a)
    return dict(D=D, v_g=v_g, v_ep=v_ep, v_mag=v_mag, mu_ep=mu_ep, ep_method=ep_method)

### Какие именно данные заданы?

Одинаковый гидродинамический диаметр 100 нм выбран для контролируемого сравнения веществ, а ζ каждого вещества задан **отдельно** как возможный результат измерения в выбранном водном электролите (это не справочные константы). `ionic_strength_M=0.02` соответствует допущению об одном моновалентном электролите для оценки длины Дебая. Плотности относятся к условным сплошным сферам. Для магнитного случая `chi_eff=0.2` — *эффективный иллюстративный параметр* линейной восприимчивости, который в опыте нужно определять по намагничиванию, а не подменять одним только названием Fe₃O₄. Магнитное поле отключено у Au и SiO₂ через `chi_eff=0`.

Если гидродинамический размер получен DLS, а масса — от меньшего твёрдого ядра, формула оседания со сплошной сферой даст неверный результат. Запишите отдельно радиус ядра, оболочку и плавучую массу; не используйте плотность массивного Au или Fe₃O₄ для пористого агрегата.

In [ ]:
base = dict(
    diameter_nm=100.0, ethanol_frac=0.0, T_K=298.15,
    eta_Pa_s=0.00089, rho_medium_kg_m3=997.0,
    epsilon_r=78.3, ionic_strength_M=0.020,
)
systems = [
    System(material="Au", zeta_mV=-25.0, rho_particle_kg_m3=19300., **base),
    System(material="Fe₃O₄", zeta_mV=-30.0, rho_particle_kg_m3=5200., chi_eff=0.2, **base),
    System(material="SiO₂", zeta_mV=-40.0, rho_particle_kg_m3=2200., **base),
]
print("Материал | d_h, нм | ζ, мВ | этанол, объёмная доля | T, K | D, мкм²/с | оседание, мкм/с")
for s in systems:
    c = coefficients(s, Fields())
    print(f"{s.material:7} | {s.diameter_nm:7.1f} | {s.zeta_mV:6.1f} | {s.ethanol_frac:6.2f}"
          f" | {s.T_K:6.2f} | {c['D']*1e12:10.3f} | {c['v_g']*1e6:8.4f}")

# При равных гидродинамических размерах D не содержит плотности или ζ.
assert np.allclose([diffusion_m2_s(s) for s in systems], diffusion_m2_s(systems[0]))
print("Вода, оценка λ_D:", round(debye_length_m(systems[0])*1e9, 2), "нм")

### Электрическое и магнитное воздействие

Для электрического поля сначала предпочтительнее **измеренная** электрофоретическая подвижность `measured_mu_ep_m2_V_s`. Только если она отсутствует, код использует $\mu_{ep}=\varepsilon_0\varepsilon_r\zeta/\eta$ в приближении тонкого двойного слоя. Программный порог $a/\lambda_D\ge20$ — *учебное правило остановки*, не гарантия точности: релаксация ионов, поверхностная проводимость, состав растворителя и покрытие частиц тоже могут нарушить формулу. В водно-спиртовой смеси поле $E\ne0$ без измеренной подвижности вызывает ошибку. ζ и подвижность одного образца нельзя переносить в другой состав среды как неизменные параметры.

Для магнетита локальная приближённая сила $F_m=V\chi_{eff}B(\partial B/\partial x)/\mu_0$. Постоянное пространственное поле без градиента даёт $F_m=0$, хотя оно может менять ориентацию или способствовать сборке **взаимодействующих** частиц. При изменении $B$ вдоль траектории или насыщении требуется поле $B(x)$ и измеренная кривая намагничивания; здесь используется локальный диапазон, где $|Gx|\ll |B|$. Параметр `chi_eff` у Au и SiO₂ равен нулю только в принятом сравнении, что не является утверждением об отсутствии всякой диамагнитной силы в точной модели.

In [ ]:
flow = Fields(u0_m_s=0.8e-6)
electric = Fields(u0_m_s=0.8e-6, E_x_V_m=20.0)
magnetic = Fields(u0_m_s=0.8e-6, B_T=0.05, gradient_T_m=30.0)

print("Режим | материал | u, мкм/с | v_E, мкм/с | v_B, мкм/с | v_g, мкм/с | формула для E")
for label, field in [("поток", flow), ("E", electric), ("B", magnetic)]:
    for s in systems:
        c = coefficients(s, field)
        print(f"{label:6} | {s.material:7} | {field.u0_m_s*1e6:8.3f}"
              f" | {c['v_ep']*1e6:8.3f} | {c['v_mag']*1e6:8.3f}"
              f" | {c['v_g']*1e6:8.4f} | {c['ep_method']}")
assert coefficients(systems[1], Fields(B_T=0.05))["v_mag"] == 0
assert coefficients(systems[1], Fields(gradient_T_m=30))["v_mag"] == 0

## Ансамбль и контроль случайного генератора

Выполняем независимые численные траектории при фиксированном seed. Их количество уменьшает шум Монте-Карло для **заданных** параметров, но не оценивает разброс ζ, размера и свойств среды между реальными синтезами. В архив не сохраняем все шаги всех частиц: ниже хранятся только выбранные моменты, моменты распределения и 12 иллюстративных траекторий. При свободной диффузии и постоянном дрейфе приращения дают точное распределение на узлах сетки; при пространственно меняющемся потоке проверяйте чувствительность к $\Delta t$ и область локального приближения.

In [ ]:
def simulate(s: System, f: Fields, *, t_final_s=20.0, dt_s=0.1,
             n_particles=2500, seed=42, every=10):
    steps_float = t_final_s / dt_s
    if not np.isclose(steps_float, round(steps_float)):
        raise ValueError("t_final_s / dt_s должно быть целым")
    steps = round(steps_float)
    if steps <= 0 or n_particles <= 1 or every <= 0:
        raise ValueError("Проверьте число шагов и частиц")
    c = coefficients(s, f)
    rng = np.random.default_rng(seed)
    r = np.zeros((n_particles, 2), dtype=float)  # метры
    times = [0.0]
    means = [np.zeros(2)]
    centered_msd = [0.0]
    tracks = [r[:12].copy()]
    sigma = np.sqrt(2 * c["D"] * dt_s)
    for step in range(1, steps + 1):
        # Шум в обоих направлениях независим; поток рассчитывается при y_n.
        noise = sigma * rng.standard_normal((n_particles, 2))
        u_x = f.u0_m_s + f.shear_s_inv * r[:, 1]
        r[:, 0] += (u_x + c["v_ep"] + c["v_mag"]) * dt_s + noise[:, 0]
        r[:, 1] += c["v_g"] * dt_s + noise[:, 1]
        if step % every == 0 or step == steps:
            m = r.mean(axis=0)
            times.append(step * dt_s)
            means.append(m.copy())
            centered_msd.append(np.mean(np.sum((r - m)**2, axis=1)))
            tracks.append(r[:12].copy())
    return dict(t=np.array(times), mean=np.array(means),
                centered_msd=np.array(centered_msd), tracks=np.array(tracks),
                final=r.copy(), coeff=c)


# Предельная проверка без дрейфа и стенок: два независимых гауссовых измерения.
s = systems[2]
check = simulate(s, Fields(), n_particles=5000, seed=123)
t = check["t"][-1]
D = check["coeff"]["D"]
sigma_mean = np.sqrt(2 * D * t / len(check["final"]))
assert np.all(np.abs(check["mean"][-1]) < 5 * sigma_mean)
assert abs(check["centered_msd"][-1] - 4 * D * t) < 0.08 * 4 * D * t
print(f"Контроль: D={D*1e12:.3f} мкм²/с, 4Dt={4*D*t*1e12:.1f} мкм²,"
      f" симуляция={check['centered_msd'][-1]*1e12:.1f} мкм²")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(check["t"], check["centered_msd"]*1e12, label="Ансамбль, центрированный MSD")
ax.plot(check["t"], 4*D*check["t"]*1e12, "--", label="Теория: 4Dt")
ax.set(xlabel="Время, с", ylabel="MSD в 2D, мкм²", title="Проверка свободной диффузии")
ax.legend(); ax.grid(alpha=.25)
plt.show()

## Сравнение трёх материалов в трёх полях

Точки на рисунке — среднее по ансамблю; горизонтальные отрезки показывают **только** ошибку Монте-Карло среднего при фиксированных параметрах ($\pm2$ стандартные ошибки), а кресты — теорию постоянного дрейфа. Это не экспериментальные доверительные интервалы. Для чисто магнитного случая отклик в данном приближении есть только у Fe₃O₄; для электрического поля он возможен у всех трёх дисперсий, потому что для каждой отдельно задан ζ в **той же** водной среде. Изменение ζ влияет на электрофорез, а не на Stokes–Einstein в разбавленном пределе.

In [ ]:
cases = [("поток", flow), ("электрическое поле", electric),
         ("магнитный градиент", magnetic)]
results = {}
fig, ax = plt.subplots(figsize=(9, 4.5))
offsets = np.array([-.24, 0, .24])
colors = ["#c2911b", "#6d5d95", "#287e94"]
for j, (case_name, f) in enumerate(cases):
    for i, s in enumerate(systems):
        out = simulate(s, f, seed=2026+i+j*20)
        results[(case_name, s.material)] = out
        n = len(out["final"])
        t = out["t"][-1]
        c = out["coeff"]
        theory_x_um = (f.u0_m_s + c["v_ep"] + c["v_mag"]) * t * 1e6
        mean_x_um = out["mean"][-1, 0] * 1e6
        se_x_um = np.sqrt(2*c["D"]*t/n) * 1e6
        assert abs(mean_x_um - theory_x_um) < 5*se_x_um
        x = j + offsets[i]
        ax.errorbar(x, mean_x_um, yerr=2*se_x_um, fmt="o", color=colors[i],
                    capsize=3, label=s.material if j == 0 else None)
        ax.plot(x, theory_x_um, "k+", ms=8)
ax.set_xticks(range(3), [name for name, _ in cases])
ax.set(xlabel="Режим", ylabel="Среднее смещение x за 20 с, мкм",
       title="Точки: ансамбль; чёрный +: расчёт постоянного дрейфа")
ax.grid(axis="y", alpha=.25); ax.legend(title="Материал")
fig.tight_layout(); plt.show()

# Пример 12 отдельных траекторий для Fe₃O₄ при магнитном поле.
tr = results[("магнитный градиент", "Fe₃O₄")]["tracks"] * 1e6
fig, ax = plt.subplots(figsize=(6, 5))
for p in range(tr.shape[1]):
    ax.plot(tr[:, p, 0], tr[:, p, 1], lw=.8, alpha=.65)
ax.set(xlabel="x, мкм", ylabel="y, мкм", title="Отдельные пути Fe₃O₄; масштаб дрейфа и шума")
ax.grid(alpha=.2); fig.tight_layout(); plt.show()

### Неоднородный локальный поток

Для $u_x(y)=u_0+\dot\gamma y$ при начальном $y=0$ и отсутствии стенок известны $\langle x\rangle=u_0t+\tfrac12\dot\gamma v_g t^2$ и $\mathrm{MSD}_{2D}=4Dt+\tfrac23D\dot\gamma^2t^3$ после центрирования. Кубический вклад описывает дисперсию в сдвиге и **не должен** подменяться увеличенным постоянным коэффициентом $D$. Это локальная часть профиля скорости: как только частицы достигают стенки или выходят за область, где приближение линейно, добавьте геометрию, граничные условия и проверьте шаг времени.

In [ ]:
shear = Fields(u0_m_s=0.8e-6, shear_s_inv=0.020)
ss = systems[2]
shear_run = simulate(ss, shear, n_particles=5500, seed=2027)
tt = shear_run["t"]
cc = shear_run["coeff"]
predicted_mean_x = shear.u0_m_s*tt + 0.5*shear.shear_s_inv*cc["v_g"]*tt**2
predicted_msd = 4*cc["D"]*tt + (2/3)*cc["D"]*shear.shear_s_inv**2*tt**3
assert abs(shear_run["mean"][-1, 0]-predicted_mean_x[-1]) < 5*np.sqrt(predicted_msd[-1]/len(shear_run["final"]))
assert abs(shear_run["centered_msd"][-1]-predicted_msd[-1]) < .09*predicted_msd[-1]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tt, shear_run["centered_msd"]*1e12, label="Сдвиг: симуляция")
ax.plot(tt, predicted_msd*1e12, "--", label="Сдвиг: 4Dt + (2/3)Dγ²t³")
ax.plot(tt, 4*cc["D"]*tt*1e12, ":", label="Свободная диффузия: 4Dt")
ax.set(xlabel="Время, с", ylabel="Центрированный MSD в 2D, мкм²",
       title="Дисперсия в локальном сдвиговом потоке")
ax.legend(); ax.grid(alpha=.25); fig.tight_layout(); plt.show()

## Водная и водно-спиртовая среда

Ниже `ethanol_frac=0.5` означает пример приготовления из равных объёмов воды и этанола. **Вязкость 2.5 мПа·с, плотность 910 кг/м³, $\varepsilon_r=55$ и ζ для этой среды — условные вводимые значения**, а не табличные данные смеси. Соотношение объёмов до смешения не равно автоматически объёмной доле в конечном объёме, и свойства смеси нелинейны. Для реального расчёта укажите протокол смешения, измерьте/возьмите согласованные параметры при заданной $T$, определите pH/электролит и заново измерьте гидродинамический размер и ζ. На этапе испарения состав меняется во времени и этот стационарный пример применять нельзя.

In [ ]:
mixture = replace(
    systems[2], ethanol_frac=0.5, zeta_mV=-18.0,  # условное повторное измерение
    eta_Pa_s=0.0025, rho_medium_kg_m3=910.0, epsilon_r=55.0,
    ionic_strength_M=0.0,  # ионный состав не установлен для смешанной среды
)
for medium in (systems[2], mixture):
    c = coefficients(medium, flow)
    print(f"{medium.material}; этанол={medium.ethanol_frac:.1f}; T={medium.T_K:.2f} K;"
          f" η={medium.eta_Pa_s*1e3:.2f} мПа·с; ζ={medium.zeta_mV:.0f} мВ;"
          f" D={c['D']*1e12:.3f} мкм²/с")
assert diffusion_m2_s(mixture) < diffusion_m2_s(systems[2])
try:
    coefficients(mixture, electric)
except ValueError as exc:
    print("Ожидаемая остановка без измеренной μ для смеси:", exc)
else:
    raise AssertionError("Смешанная среда не должна использовать водную формулу автоматически")

# При необходимости после измерения μ: replace(mixture, measured_mu_ep_m2_V_s=...)

## Что изменить самостоятельно

1. **Размер и состав.** Задайте два размера Au и SiO₂, затем новые свойства среды при другой температуре. Предскажите отношения $D$, $|v_g|$ и ширины распределения *до* запуска; покажите зависимость от $d_h$, $T$ и $\eta$. Объясните, почему нельзя менять только `ethanol_frac` при неизменной вязкости и ζ.
2. **Поля.** Поменяйте знак $E_x$ и знак $\partial B/\partial x$ отдельно. Сопоставьте изменение среднего смещения с размером случайного облака $\sqrt{4Dt}$. Покажите, что для Fe₃O₄ при $G=0$ направленная магнитная скорость равна нулю.
3. **Режим электрофореза.** Уменьшите ионную силу так, чтобы `a/λ_D < 20`. Почему код останавливается? Введите измеренную `measured_mu_ep_m2_V_s` с источником и неопределённостью измерения либо выберите обоснованную электрокинетическую модель; не трактуйте ζ как свободный полный заряд сферы.
4. **Проверка модели на опыте.** Предложите независимое измерение $D$ (например, по траекториям или DLS), ζ/подвижности и профиля поля. Укажите, какая геометрия, концентрация, вязкость, свойства поля, температура и независимые приготовления понадобятся, чтобы оценить пригодность прогноза. Одни лишь повторные траектории из той же дисперсии не заменяют независимые партии.

**За пределами примера:** агрегирование, электростатические парные взаимодействия, высыхание и формирование структур, испарительная конвекция, искажение поля у стенок, полидисперсность и магнитное насыщение. Эти вопросы вынесены в отдельный практикум о сборке SiO₂. В случае сдвигового потока за пределами локального участка нужно добавить стены и корректные условия отражения/осаждения.

**Физические источники для дальнейшего чтения:** [исследование электрофореза в пределе тонкого двойного слоя, *J. Fluid Mech.*](https://www.cambridge.org/core/journals/journal-of-fluid-mechanics/article/shapedependence-of-electrophoretic-mobility-an-aiassisted-perturbation-analysis/236DA6D39FF5242DFFD12403A90A6CEB); [магнитофорез наночастиц, *ACS Nano*](https://pubs.acs.org/doi/10.1021/nn102383s); [линейный и насыщенный магнитный отклик, *Lab on a Chip*](https://pubs.rsc.org/en/content/articlehtml/2026/lc/d5lc01072a); [корреляция вязкости чистого этанола, NIST](https://www.nist.gov/publications/reference-correlation-viscosity-ethanol-triple-point-620-k-and-102-mpa). Выделенные численные значения свойств **модельные**, они не взяты из этих статей как данные конкретного опыта.